# 1. Download and clean structures from the RCSB PDB

## What is the goal of this notebook?

You can use this notebook to learn how to run `CIFcurator_Download.py`. This script downloads all the structures deposited in the PDB that match a 'keyword', and then 'cleans' them, so that they are ready for Step 3. For example, in this tutorial, we will download all the structures that contain the keyword 'triosephosphate isomerase' in their metadata.

Optional: the script can also download the [computed structure models](https://www.rcsb.org/docs/general-help/computed-structure-models-and-rcsborg) (CSMs), predicted by AlphaFold, that the RCSB PDB also offers. In this tutorial, we donwload 10 CSMs.


*NOTE: sequences and structures are downloaded from the [wwPDB beta archive](https://www.wwpdb.org/ftp/pdb-beta-ftp-sites), which will replace the current archive on July 21, 2027. It uses the extended PDB IDs: the legacy ID `1aw1` becomes `pdb_00001aw1`, and the files are named after it (e.g. `pdb_00001aw1+01.cif`).*

## How to run `CIFcurator_Download.py`

In the terminal, type:

> CIFcurator_Download.py `"keyword"` --projdir `{Project Directory}` --csm `{True or False}`

Use quotes when the keyword has more than one word, e.g. `"triosephosphate isomerase"`.

| Argument | What it does | Default |
|---|:---|:---|
| `keyword` | The keyword to search for in the PDB (required) | |
| `--projdir` | Project directory. The directories `raw_bank` and `clean_bank` are created inside it | current directory |
| `--csm` | `True` to also download computed structure models (CSMs) | `False` |
| `--max-models` | With `--csm True`, download at most this number of CSMs, best matches first | all of them |
| `--seqfile` | File with the sequences of all the structures in the PDB | `{projdir}/seqres.txt` |
| `--update` | Download the sequence file again, even if it already exists | |

The option `-h` prints the help of the script:

In [1]:
! CIFcurator_Download.py -h

usage: CIFcurator_Download.py [-h] [--projdir PROJDIR] [--csm {True,False}]
                              [--max-models N] [--seqfile SEQFILE] [--update]
                              keyword

Step 1 of CIFcurator: download the structures that match a keyword from the PDB, and clean them.

positional arguments:
  keyword             Keyword to search for in the PDB, e.g. "triosephosphate
                      isomerase" (in quotes)

options:
  -h, --help          show this help message and exit
  --projdir PROJDIR   Project directory, where raw_bank and clean_bank are
                      created (default: current directory)
  --csm {True,False}  Also download computed structure models (CSMs) from
                      AlphaFold DB (default: False)
  --max-models N      With --csm True, download at most N CSMs, best matches
                      first (default: all of them)
  --seqfile SEQFILE   File with the sequences of all the structures in the PDB
                      (default: {

## What does the script do?

### 1. Download the sequences of all the structures in the PDB

The script downloads the sequences of all the structures currently deposited in the PDB (`pdb_seqres.fasta.gz` from the [wwPDB beta archive](https://www.wwpdb.org/ftp/pdb-beta-ftp-sites), decompressed) to `{projdir}/seqres.txt`. Depending on your internet connection, it may take a few minutes.

You only need to do this once: if the file already exists, the script uses it. To update the list of sequences, add `--update`. To use a file you downloaded before, for example for another project, give its path with `--seqfile`.

### 2. Retrieve the reference sequence(s) from the keyword

The script searches for all the sequences whose metadata matches the keyword (`pdbutils.retrieve_sequence_from_PDB` with `mode='metadata'`). The number of hits is printed to screen.

### 3. Retrieve all the sequences that match the reference sequence(s)

The script uses the sequences found in the previous search to find all the chains in the PDB whose sequence matches one of them: a chain matches when its sequence ends with one of the reference sequences (`pdbutils.retrieve_sequences_matching_from_PDB`). This finds structures that contain the molecule of interest, even if their metadata do not have the keyword. All the sequences retrieved are printed to screen.

### 4. Download the PDB files

The script downloads the biological assemblies of the structures found in the previous step from the wwPDB beta archive into `{projdir}/raw_bank` (`pdbutils.download_pdb_from_metadata`). Each assembly is saved as `PDBID+NN.cif`, where PDBID is the extended PDB ID and NN the assembly number (e.g. `pdb_00001aw1+01.cif` to `pdb_00001aw1+04.cif`).

### 5. Optional: download computed structure models (CSMs)

Besides experimental structures, RCSB PDB also offers [computed structure models](https://www.rcsb.org/docs/general-help/computed-structure-models-and-rcsborg) (CSMs), predicted by AlphaFold. **This step is optional**, and it only runs with `--csm True`.

The script finds the CSMs that match the keyword with the RCSB full text search (the same search as the RCSB PDB web site, so it can find a different set of proteins than the search above), and downloads their mmCIF files from [AlphaFold DB](https://alphafold.ebi.ac.uk) into the same 'raw_bank' as the experimental structures (`pdbutils.download_csm_from_keyword`). Each model is saved as `CSMID+01.cif`, where CSMID is the RCSB ID of the model (e.g. `AF_AFP60174F1+01.cif`). Models from other databases (ModelArchive, IDs starting with `MA_`) are skipped.

A search can find many models (more than 700 for triosephosphate isomerase), so `--max-models` limits how many are downloaded, best matches first. In this tutorial we download only 10 models (`--max-models 10`). **To download all the models that match the keyword, leave out `--max-models`.**

**Before including CSMs, keep in mind:**
* A CSM contains **one protein chain only**: no nucleic acids, ligands, waters or other copies of the protein. For example, a Cas9 model has no guide RNA or target DNA, and a triosephosphate isomerase model is a monomer, while the experimental structures are dimers.
* A CSM covers the full UniProt sequence, including parts that are usually missing from experimental structures, such as signal peptides.
* The B-factor column of a CSM holds the model confidence (pLDDT), not a B-factor.
* The entity name of a CSM comes from UniProt, and can differ from the names in the experimental structures (e.g. `TRIOSEPHOSPHATE ISOMERASE, CYTOSOLIC`). In Step 3 each name is a separate entity that needs its own chain IDs.

### 6. Clean the structures: one CIF file per biological assembly

Finally, the script 'cleans' the CIF files of 'raw_bank', and saves them in `{projdir}/clean_bank` (`cleanutils.process` with `step='clean'`):

1. **One file per [biological assembly](https://pdb101.rcsb.org/learn/guide-to-understanding-pdb-data/biological-assemblies#Anchor-BioUnit).** Each file downloaded already holds one complete assembly, including the copies of chains generated by symmetry (their chain IDs have a suffix, e.g. `A-2`). Therefore, each file is rewritten whole and keeps its name. Assembly files list the PDB ID as `XXXX`, so `_entry.id` is taken from the file name.

   *Full PDB entries, as downloaded by earlier versions of CIFcurator, are still split into one file per assembly: `PDBID+00.cif` if the entry has a single assembly, `PDBID+01.cif`, `PDBID+02.cif`, ... otherwise.*

2. **Only a limited set of categories is kept**, and they are written in a standard way:
   - `entry`, `citation_author`, `citation`, `exptl` (experimental method; not written for computed structure models, which have none) and `entity`. Entity names are written in upper case, without colons or single quotes.
   - The categories that the coordinates refer to: `entity_poly`, `entity_poly_seq`, `chem_comp`, `struct_asym`, `pdbx_poly_seq_scheme` and `atom_type`.
   - `atom_site`, the coordinates, with the 20 columns that Steps 3-5 read.

3. **The files stay consistent with the mmCIF dictionary.** Each file declares the dictionary it follows (`mmcif_pdbx.dic`, version 5.418, set by `MMCIF_DICT_VERSION` in `cleanutils.py`) in its `_audit_conform` category. The key items of the categories written are included (`citation.id`, `citation_author.citation_id`, `citation_author.ordinal`, `exptl.entry_id`). Steps 3-5 keep `struct_asym` and `pdbx_poly_seq_scheme` up to date when they rename chains and residues.

> **Note:** `_exptl.resolution` is not an mmCIF dictionary item, so the resolution is not written. It can be read from the source files (e.g. `_refine.ls_d_res_high`), as in the 'Analysis.SummaryPDBDataset' notebook, or from the RCSB PDB Data API, as in 'Analysis.SummaryPDBDataset_RCSBAPI'.

## Run the script for this tutorial

We will save our structures in the project directory './TIM/'. The script creates it if it does not exist.

In [2]:
PROJDIR="./TIM/"

Run the next cell to download all the structures that match 'triosephosphate isomerase', and 10 computed structure models.

Note: Depending on your internet connection, it may take a while to download all the structures.

**Note:** We recommend running the script directly on the terminal. We are running it from the notebook just for demonstration purpose.

In [3]:
! CIFcurator_Download.py "triosephosphate isomerase" --projdir $PROJDIR --csm True --max-models 10

wrote ./TIM//seqres.txt from https://files-beta.wwpdb.org/pub/wwpdb/pdb/derived_data/pdb_seqres.fasta.gz
642 sequences were identified as potential hits!
644 sequences were retrieved!
>pdb_00001ag1_O mol:protein length:250  TRIOSEPHOSPHATE ISOMERASE MSKPQPIAAANWKCNGSQQSLSELIDLFNSTSINHDVQCVVASTFVHLAMTKERLSHPKFVIAAQNAIAKSGAFTGEVSLPILKDFGVNWIVLGHSERRAYYGETNEIVADKVAAAVASGFMVIACIGETLQERESGRTAVVVLTQIAAIAKKLKKADWAKVVIAYEPVWAIGTGKVATPQQAQEAHALIRSWVSSKIGADVAGELRILYGGSVNGKNARTLYQQRDVNGFLVGGASLKPEFVDIIKATQ
>pdb_00001ag1_T mol:protein length:250  TRIOSEPHOSPHATE ISOMERASE MSKPQPIAAANWKCNGSQQSLSELIDLFNSTSINHDVQCVVASTFVHLAMTKERLSHPKFVIAAQNAIAKSGAFTGEVSLPILKDFGVNWIVLGHSERRAYYGETNEIVADKVAAAVASGFMVIACIGETLQERESGRTAVVVLTQIAAIAKKLKKADWAKVVIAYEPVWAIGTGKVATPQQAQEAHALIRSWVSSKIGADVAGELRILYGGSVNGKNARTLYQQRDVNGFLVGGASLKPEFVDIIKATQ
>pdb_00001aw1_A mol:protein length:256  TRIOSEPHOSPHATE ISOMERASE MRHPVVMGNWKLNGSKEMVVDLLNGLNAELEGVTGVDVAVAPPALFVDLAERTLTEAGSAIILGAQNTDLNNSGAFTGDMSPAMLKEFGATHIIIGHSERREYHAESDEFVAKKFA

## Inspect the results

The downloaded structures are in 'raw_bank', and the clean structures, ready for Step 3, are in 'clean_bank'. Both directories should have the same CIF files:

In [4]:
! ls $PROJDIR/raw_bank/*.cif | wc -l
! ls $PROJDIR/clean_bank/*.cif | wc -l

     384
     384


In [5]:
! ls $PROJDIR/clean_bank

AF_AFB5YL53F1+01.cif pdb_00002v5l+02.cif  pdb_00005bmx+02.cif
AF_AFB8E2R2F1+01.cif pdb_00002vei+01.cif  pdb_00005bnk+01.cif
AF_AFK7UUB0F1+01.cif pdb_00002vek+01.cif  pdb_00005brb+01.cif
AF_AFO02611F1+01.cif pdb_00002vek+02.cif  pdb_00005csr+01.cif
AF_AFO27120F1+01.cif pdb_00002vel+01.cif  pdb_00005csr+02.cif
AF_AFQ12UK2F1+01.cif pdb_00002vel+02.cif  pdb_00005css+01.cif
AF_AFQ58923F1+01.cif pdb_00002vem+01.cif  pdb_00005css+02.cif
AF_AFQ5JHG3F1+01.cif pdb_00002vem+02.cif  pdb_00005eyw+01.cif
AF_AFQ869R8F1+01.cif pdb_00002ven+01.cif  pdb_00005gv4+01.cif
AF_AFQ9UWN5F1+01.cif pdb_00002ven+02.cif  pdb_00005gzp+01.cif
info.txt             pdb_00002vfd+01.cif  pdb_00005i3f+01.cif
pdb_00001ag1+01.cif  pdb_00002vfe+01.cif  pdb_00005i3f+02.cif
pdb_00001aw1+01.cif  pdb_00002vff+01.cif  pdb_00005i3g+01.cif
pdb_00001aw1+02.cif  pdb_00002vfg+01.cif  pdb_00005i3g+02.cif
pdb_00001aw1+03.cif  pdb_00002vfg+02.cif  pdb_00005i3h+01.cif
pdb_00001aw1+04.cif  pdb_00002vfh+01.cif  pdb_00005i3i+01.cif
pdb_0000

The next step is to assign the chain IDs of the entities found in the clean CIF files, with `CIFcurator_MolID.py` (Step 3), using 'clean_bank' as the input directory.